In [1]:
using Revise
using InteractiveUtils

includet("phase0/functions/load_phase0.jl")


In [2]:
df, meta = load_dataframes();

QoG Time-Series Loader Pipeline
Input: data/qog_std_ts_jan25.arrow

>>> Step 1/5: Loading raw data with identity promotion...
    Loaded: 12391 rows × 2010 columns
    ✓ ggis_rowid assigned

>>> Step 2/5: Previewing rescue collisions...
>>> RESCUE COLLISION PREVIEW (informational — NO rows will be deleted):
    Total (ccode, year) pairs with >1 row: 22

    By entity combination:
      VDR + VNM: 22 years (1955-1976)
        Years: 1955, 1956, 1957, 1958, 1959, 1960, 1961, 1962, 1963, 1964, 1965, 1966, 1967, 1968, 1969, 1970, 1971, 1972, 1973, 1974, 1975, 1976

    NOTE: Use `ggis_rowid` as unique key, or (ident_ccode, ident_year, ident_ccodealp)
    ⚠️  Found 1 collision group(s)
    (See year details above)

>>> Step 3/5: Rescuing historical ccodes...
>>> Historical Ccode Rescue:
    Missing before: 234
    Rescued: 234
    Missing after: 0
    Row count: 12391 (unchanged)
    By alpha code:
      ETH → 231: 47 rows
      YEM → 887: 44 rows
      DEU → 276: 42 rows
      MHL → 584: 3

In [3]:
dataframe_summaries()

=== DataFrame: REGION_LABELS ===
10×2 DataFrame

=== DataFrame: df ===
12391×2013 DataFrame

=== DataFrame: meta ===
2010×16 DataFrame



In [4]:

# INTENT: Build country-level feature table (one row per ident_ccode) with period means,
# │          delta_total, recent_change, volatility, and period missrates per slug.
# │  USE WHEN: Step 1 of clustering prep — you need country-level summary for clustering.
country_features_df, audit = build_country_features_df(df, meta);
Arrow.write("country_features.arrow", country_features_df)
Arrow.write("country_features_audit.arrow", audit)

"country_features_audit.arrow"

In [5]:

# Load
country_features_df = DataFrame(Arrow.Table("country_features.arrow"));
audit = DataFrame(Arrow.Table("country_features_audit.arrow"));

In [6]:
size(audit)

(2009, 5)

In [7]:
size(country_features_df)

(200, 22101)

In [8]:
first(country_features_df, 5);

In [9]:

# New
miss_df = build_country_period_missingness_baseline_df(country_features_df, meta);


Step 2A — Missingness baseline diagnostics
Periods: P1, P2, P3, P4
Global slugs in metadata:   616
Regional slugs in metadata: 83
Global missrate columns found (by period):
  P1: 616
  P2: 616
  P3: 616
  P4: 616
Regional pool sizes (by period, region):
  P1  r=1  n=0
  P1  r=2  n=0
  P1  r=3  n=0
  P1  r=4  n=0
  P1  r=5  n=0
  P1  r=6  n=0
  P1  r=7  n=0
  P1  r=8  n=0
  P1  r=9  n=0
  P1  r=10  n=0
  P2  r=1  n=0
  P2  r=2  n=0
  P2  r=3  n=0
  P2  r=4  n=0
  P2  r=5  n=0
  P2  r=6  n=0
  P2  r=7  n=0
  P2  r=8  n=0
  P2  r=9  n=0
  P2  r=10  n=0
  P3  r=1  n=0
  P3  r=2  n=0
  P3  r=3  n=0
  P3  r=4  n=83
  P3  r=5  n=0
  P3  r=6  n=0
  P3  r=7  n=0
  P3  r=8  n=0
  P3  r=9  n=0
  P3  r=10  n=0
  P4  r=1  n=0
  P4  r=2  n=0
  P4  r=3  n=0
  P4  r=4  n=83
  P4  r=5  n=0
  P4  r=6  n=0
  P4  r=7  n=0
  P4  r=8  n=0
  P4  r=9  n=0
  P4  r=10  n=0



In [13]:
meta.ggis_geo_classification

meta[meta.ggis_geo_classification .== "regional", :]

LoadError: ArgumentError: unable to check bounds for indices of type Missing

In [14]:
meta[ismissing.(meta.ggis_geo_classification), :]

Row,slug,prefix,label,description,type,provenance,min_year,max_year,ggis_birth_year,ggis_death_year,ggis_is_active,ggis_temporal_gap,ggis_temporal_profile,ggis_global_penetration,ggis_geo_classification,ggis_region_penetration
,String31,String7,String?,String,String15?,String,Int64?,Int64?,Int64?,Int64?,Bool?,Bool?,String15?,Float64?,String15?,String?
1,who_roadtrd,who,missing,Estimated road traﬃc death rate (per 100 000 population),continuous,SURVEY,2000,2019,missing,missing,missing,missing,missing,missing,missing,missing


In [17]:
propertynames(meta)

16-element Vector{Symbol}:
 :slug
 :prefix
 :label
 :description
 :type
 :provenance
 :min_year
 :max_year
 :ggis_birth_year
 :ggis_death_year
 :ggis_is_active
 :ggis_temporal_gap
 :ggis_temporal_profile
 :ggis_global_penetration
 :ggis_geo_classification
 :ggis_region_penetration

In [20]:
using DataFrames

subset(meta, :ggis_geo_classification => ByRow(isequal("regional")))[!, [:slug, :min_year, :max_year, :ggis_global_penetration, :ggis_geo_classification, :ggis_region_penetration]]

Row,slug,min_year,max_year,ggis_global_penetration,ggis_geo_classification,ggis_region_penetration
,String31,Int64?,Int64?,Float64?,String15?,String?
1,aii_acc,2013,2022,0.177369,regional,"[0.0, 0.0, 0.25, 0.9795918367346939, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]"
2,aii_aio,2013,2022,0.177369,regional,"[0.0, 0.0, 0.25, 0.9795918367346939, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]"
3,aii_cilser,2013,2022,0.177369,regional,"[0.0, 0.0, 0.25, 0.9795918367346939, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]"
4,aii_elec,2013,2022,0.177369,regional,"[0.0, 0.0, 0.25, 0.9795918367346939, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]"
5,aii_pubm,2013,2022,0.177369,regional,"[0.0, 0.0, 0.25, 0.9795918367346939, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]"
6,aii_q01,2013,2017,0.162779,regional,"[0.0, 0.0, 0.25, 0.9795918367346939, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]"
7,aii_q02,2013,2022,0.177369,regional,"[0.0, 0.0, 0.25, 0.9795918367346939, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]"
8,aii_q03,2013,2022,0.177369,regional,"[0.0, 0.0, 0.25, 0.9795918367346939, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]"
9,aii_q04,2013,2022,0.177369,regional,"[0.0, 0.0, 0.25, 0.9795918367346939, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]"


In [9]:

# New
anchors_df = build_country_period_anchor_df(df);


Step 2A — Anchor diagnostics (post-cluster interpretation)
Anchors:
  POP:    wpp_pop
  WEALTH: gle_cgdpc (pre-1960), wdi_gdpcapcon2015 (>= 1960)
  GOV:    bmr_dem
Rows produced: 800
Countries: 200



In [10]:

# New
presence_df = build_other_slug_presence_df(df, meta);  # uses PRESENT_MIN_OBS_OTHER = 2


Step 2A — Other-slug presence diagnostics
Other slugs in metadata: 1311
Other slugs found in df: 1310
min_present_obs:         2
Sparse presence rows:    311939
Example present rows:
10×4 DataFrame
 Row │ slug       ident_ccode  period  nonmissing_obs 
     │ String     Int64        String  Int64          
─────┼────────────────────────────────────────────────
   1 │ aid_crnc             4  P1                   2
   2 │ aid_crnc             4  P2                   2
   3 │ aid_crnc             4  P3                   2
   4 │ aid_crnio            4  P1                   2
   5 │ aid_crnio            4  P2                   2
   6 │ aid_crnio            4  P3                   2
   7 │ aid_crsc             4  P1                   2
   8 │ aid_crsc             4  P2                   2
   9 │ aid_crsc             4  P3                   2
  10 │ aid_crsio            4  P1                   2



# Step 3A

In [11]:

# New
X, slug_index, cell_index = build_slug_cell_matrix(presence_df);


Step 3A — Membership topology matrix build
Input presence rows: 311939
Unique Other slugs:  1307
Unique cells:        775  (country,period)
nnz(X):              311939
Density:             30.7959%

Slug prevalence (#cells present):
  min/median/max: 1 / 143 / 775
Cell load (#slugs present):
  min/median/max: 17 / 350 / 972

Example slugs: aid_cpnc, aid_cpsc, aid_crnc, aid_crnio, aid_crsc, aid_crsio, aii_q24, ajr_settmort, bl_asyf, bl_asym
Example cells: (4, "P1"), (4, "P2"), (4, "P3"), (4, "P4"), (8, "P1"), (8, "P2"), (8, "P3"), (8, "P4"), (12, "P1"), (12, "P2")



In [12]:

# New
edges_dir = cosine_topk_edges(X, slug_index; k=20, min_sim=0.05);


Step 3A — Membership topology (cosine top-k edges)
Slugs (rows): 1307
Cells (cols): 775
Requested k:   20
min_sim:       0.05
Edges produced (directed top-k): 26140
Example edges:
10×3 DataFrame
 Row │ slug_i    slug_j               sim      
     │ String    String               Float64  
─────┼─────────────────────────────────────────
   1 │ aid_cpnc  aid_cpsc             1.0
   2 │ aid_cpnc  eu_sctppspop         0.696078
   3 │ aid_cpnc  oecd_pension_t1a     0.684416
   4 │ aid_cpnc  eu_scttotn           0.67787
   5 │ aid_cpnc  eu_sctrtotpmin       0.666967
   6 │ aid_cpnc  oecd_tiva_inter_t1j  0.664966
   7 │ aid_cpnc  oecd_tiva_inter_t1i  0.664966
   8 │ aid_cpnc  oecd_tiva_inter_t1g  0.664966
   9 │ aid_cpnc  oecd_tiva_inter_t1h  0.664966
  10 │ aid_cpnc  oecd_tiva_inter_t1d  0.664966



In [13]:

# New
edges_und = symmetrize_edges(edges_dir; mode=:max);


Step 3A — Symmetrize top-k edges (undirected graph)
Directed edges in:   26140
Undirected edges out:19119
Mode:                max
Reciprocal edges (%):36.72
Weight min/med/max:  0.1118 / 0.9294 / 1.0
Example undirected edges:
10×4 DataFrame
 Row │ slug_a                 slug_b               weight    n_dir 
     │ String                 String               Float64   Int64 
─────┼─────────────────────────────────────────────────────────────
   1 │ oecd_gengovdistri_t1j  oecd_gengovprod_t1a  0.894427      1
   2 │ iaep_alcc              qar_plac             0.537626      1
   3 │ lis_dc5075             lis_pr8020           1.0           2
   4 │ oecd_hourswkd_t1       oecd_pphlthxp_t1c    0.869539      1
   5 │ nelda_noel             nelda_rpae           1.0           2
   6 │ ipu_l_s                nelda_rpae           0.829642      1
   7 │ oecd_evova_t1e         oecd_socexpnd_t1a    0.855921      1
   8 │ kun_wiqrleg_all        kun_wiqrleg_full     0.818121      1
   9 │ eu_eco2gdp

In [14]:

# 11
g, slugs_g, comps = graph_components_diagnostics(edges_und);


Step 3A — Graph connectivity diagnostics
Nodes: 1307
Edges: 19119
Connected components: 2
Component size min/med/max: 40 / 654 / 1267
Largest component share (%): 96.94



In [15]:

# 12
comp_slugs = component_slug_sets(comps, slugs_g)
length.(comp_slugs)  # should show [1267, 40] in some order

2-element Vector{Int64}:
 1267
   40

In [16]:

# 13
small = argmin(length.(comp_slugs)); println(comp_slugs[small])

["wwbi_fmwrprmean", "wwbi_fmwrprmedian", "wwbi_fmwrpumean", "wwbi_fmwrpumedian", "wwbi_fsprpemp", "wwbi_fspuemp", "wwbi_meanageprpe", "wwbi_meanagepupe", "wwbi_medianageprpe", "wwbi_medianagepupe", "wwbi_paycomppr", "wwbi_paycomppu", "wwbi_prpemphi", "wwbi_prpempss", "wwbi_prpempum", "wwbi_psefemp", "wwbi_psemptot", "wwbi_psemptotf", "wwbi_psemptotm", "wwbi_psemptotr", "wwbi_psemptotu", "wwbi_psepemp", "wwbi_psepempf", "wwbi_psepempm", "wwbi_psepempr", "wwbi_psepempu", "wwbi_pupemphi", "wwbi_pupempss", "wwbi_pupempum", "wwbi_rrespripemp", "wwbi_rrespubpemp", "wwbi_sprpempn", "wwbi_sprpempp", "wwbi_sprpemps", "wwbi_sprpempt", "wwbi_spupempn", "wwbi_spupempp", "wwbi_spupemps", "wwbi_spupempt", "wwbi_tertiarypubsec"]


In [19]:

# 14
g_w, slugs_g, slug_to_idx = build_weighted_slug_graph(edges_und);

## Minimal Step 3B: finish clustering now

### Not in the how to

In [57]:
labels, history = label_propagation(g)
membership = labels;
slug_cluster_df = DataFrame(
    slug = slugs_g,
    cluster_id = membership
);
sizes = combine(groupby(slug_cluster_df, :cluster_id), nrow => :n)
sort!(sizes, :n, rev=true)
println("Number of clusters is: ", nrow(sizes))
println("Largest cluster is: ", sizes[1,end])
first(sizes, 10);

Number of clusters is: 34
Largest cluster is: 113


## Step 4

In [82]:

# 15. 
cluster_presence_df, cp_audit = cluster_presence(presence_df, slug_cluster_df)
@info "cluster_presence" cp_audit

┌ Info: cluster_presence
└   cp_audit = (n_presence_rows = 311939, n_cluster_rows = 1307, n_join_rows = 311939, n_missing_cluster_id = 0, n_cells = 10810, min_cluster_size = 16, max_cluster_size = 113)


In [83]:
enriched_df, enrich_audit = attach_environment_and_anchors(cluster_presence_df, miss_df, anchors_df)
@info "attach_environment_and_anchors" enrich_audit

┌ Info: attach_environment_and_anchors
└   enrich_audit = (n_in = 10810, n_after_miss = 10810, n_after_anchors = 10810, miss_dup_keys = 0, anchors_dup_keys = 0, expected_present = Dict{Symbol, Bool}(:log_gdp_level => 1, :log_gdp_change => 1, :log_pop_level => 1, :dem_change => 1, :log_pop_change => 1, :miss_regional => 1, :miss_global => 1, :dem_share => 1), missing_counts = (log_gdp_level = 49, log_pop_level = 74, dem_share = 3, miss_regional = 5411, miss_global = 0), missing_rates = (miss_global = 0.0, miss_regional = 0.5005550416281221, log_pop_level = 0.0068455134135060125, log_gdp_level = 0.004532839962997225, dem_share = 0.0002775208140610546), key_types = (ident_ccode = Int64, period = String, miss_ident_ccode = Int64, miss_period = String, anchors_ident_ccode = Int64, anchors_period = String))


In [71]:
cluster_summary_df, sum_audit = summarize_clusters(enriched_df)
@info "summarize_clusters" sum_audit

┌ Info: summarize_clusters
└   sum_audit = (n_clusters = 34, cluster_size_min = 65, cluster_size_max = 775)


In [73]:
top_countries_df, top_audit = top_countries_by_cluster(enriched_df; topn=15)
@info "top_countries_by_cluster" top_audit

period_cov_df, pc_audit = period_coverage_by_cluster(enriched_df)
@info "period_coverage_by_cluster" pc_audit

audit2 = join_audit(cluster_presence_df, miss_df, anchors_df)
@info "join_audit" audit2

┌ Info: top_countries_by_cluster
└   top_audit = (n_rows_in = 10810, n_rows_out = 510, topn = 15)
┌ Info: period_coverage_by_cluster
└   pc_audit = (n_rows_in = 10810, n_rows_out = 119, n_clusters = 34, n_periods = 4)
┌ Info: join_audit
└   audit2 = (miss_missing = -1, miss_total = 10810, anchors_missing = -1, anchors_total = 10810, miss_dup_keys = 0, anchors_dup_keys = 0)


In [86]:
names(enriched_df)

17-element Vector{String}:
 "cluster_id"
 "ident_ccode"
 "period"
 "present_slugs"
 "present_obs"
 "cluster_slug_count"
 "present_slug_share"
 "miss_global"
 "miss_regional"
 "n_global_used"
 "n_regional_used"
 "log_pop_level"
 "log_pop_change"
 "log_gdp_level"
 "log_gdp_change"
 "dem_share"
 "dem_change"

In [95]:
# describe(enriched_df[!, [:n_global_used]])
describe(enriched_df[coalesce.(enriched_df.period .== "P4", false), [:n_global_used]])

Row,variable,mean,min,median,max,nmissing,eltype
,Symbol,Float64,Int64,Float64,Int64,Int64,Union
1,n_global_used,520.0,520,520.0,520,0,"Union{Missing, Int64}"


In [100]:
describe(enriched_df[coalesce.(enriched_df.period .== "P4", false), [:n_regional_used]])

Row,variable,mean,min,median,max,nmissing,eltype
,Symbol,Float64,Int64,Float64,Int64,Int64,Union
1,n_regional_used,83.0,83,83.0,83,0,"Union{Missing, Int64}"


In [101]:
combine(groupby(enriched_df, :period),
        :n_regional_used => (x -> length(unique(x))) => :n_unique_regional)


Row,period,n_unique_regional
,String,Int64
1,P1,1
2,P2,1
3,P3,1
4,P4,1


In [102]:
names(miss_df)


6-element Vector{String}:
 "ident_ccode"
 "period"
 "miss_global"
 "miss_regional"
 "n_global_used"
 "n_regional_used"

In [4]:
run_cluster_analysis_samples()


  cluster_analysis.jl — Function intent and usage (Step 1: country-level features)

┌─ load_dataframes()
│  INTENT: Load the main QoG timeseries and the clustering metadata in one call.
│  USE WHEN: You need both df and meta_df for building country features.
│
│  RETURNS: (df, meta_df)
│    - df: main timeseries from load_qog_timeseries()
│    - meta_df: from PATH_METADATA_CLUSTER_INPUT (qog_metadata_plus2.csv)
│
│  USAGE:
│    df, meta = load_dataframes()
└──────────────────────────────────────────────────────────────────────────

┌─ build_country_features_df(df, meta_df; periods, min_obs_per_period, min_obs_for_vol, include_audit)
│  INTENT: Build country-level feature table (one row per ident_ccode) with period means,
│          delta_total, recent_change, volatility, and period missrates per slug.
│  USE WHEN: Step 1 of clustering prep — you need country-level summary for clustering.
│
│  ARGUMENTS:
│    df::DataFrame — panel with ident_ccode, ident_year, ggis_region, plus slug co